In [1]:
import $ivy.`org.apache.spark::spark-sql:3.5.8`

import $ivy.$

In [2]:
import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

import org.apache.spark.sql._
import org.apache.spark.sql.types._
import org.apache.spark.sql.functions._

In [3]:
val spark = SparkSession
                .builder()
                .master("local[*]")
                .appName("Files")
                .config("spark.log.level", "WARN")
                .config("spark.sql.debug.maxToStringFields", 1024)
                .getOrCreate()

import spark.implicits._

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/28 15:59:05 INFO SparkContext: Running Spark version 3.5.8
26/06/28 15:59:05 INFO SparkContext: OS info Linux, 6.12.0-211.22.1.el10_2.x86_64, amd64
26/06/28 15:59:05 INFO SparkContext: Java version 11.0.31
26/06/28 15:59:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting Spark log level to "WARN".


spark: SparkSession = org.apache.spark.sql.SparkSession@46de13f0
import spark.implicits._

In [4]:
println(s"spark.version == ${spark.version}")

spark.version == 3.5.8


## CSV

### Пример 1

In [5]:
import scala.sys.process._

"cat data/people.csv".!!

import scala.sys.process._
res5_1: String = """name;age;job
Jorge;30;Developer
Bob;32;Developer
"""

In [6]:
val path = "data/people.csv"
val df1 = spark.read.csv(path)
df1.show()

+------------------+
|               _c0|
+------------------+
|      name;age;job|
|Jorge;30;Developer|
|  Bob;32;Developer|
+------------------+



path: String = "data/people.csv"
df1: DataFrame = [_c0: string]

In [7]:
val df2 = spark.read.option("delimiter", ";").csv(path)
df2.show()

+-----+---+---------+
|  _c0|_c1|      _c2|
+-----+---+---------+
| name|age|      job|
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



df2: DataFrame = [_c0: string, _c1: string ... 1 more field]

In [8]:
val df3 = spark.read.option("delimiter", ";").option("header", "true").csv(path)
df3.show()

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



df3: DataFrame = [name: string, age: string ... 1 more field]

In [9]:
val df4 = spark.read.options(Map("delimiter"->";", "header"->"true")).csv(path)
df4.show

+-----+---+---------+
| name|age|      job|
+-----+---+---------+
|Jorge| 30|Developer|
|  Bob| 32|Developer|
+-----+---+---------+



df4: DataFrame = [name: string, age: string ... 1 more field]

### Пример 2

In [10]:
"head data/books.csv".!!

res10: String = """Name,Author,User Rating,Reviews,Price,Year,Genre
10-Day Green Smoothie Cleanse,JJ Smith,4.7,17350,8,2016,Non Fiction
11/22/63: A Novel,Stephen King,4.6,2052,22,2011,Fiction
12 Rules for Life: An Antidote to Chaos,Jordan B. Peterson,4.7,18979,15,2018,Non Fiction
1984 (Signet Classics),George Orwell,4.7,21424,6,2017,Fiction
"5,000 Awesome Facts (About Everything!) (National Geographic Kids)",National Geographic Kids,4.8,7665,12,2019,Non Fiction
A Dance with Dragons (A Song of Ice and Fire),George R. R. Martin,4.4,12643,11,2011,Fiction
A Game of Thrones / A Clash of Kings / A Storm of Swords / A Feast of Crows / A Dance with Dragons,George R. R. Martin,4.7,19735,30,2014,Fiction
A Gentleman in Moscow: A Novel,Amor Towles,4.7,19699,15,2017,Fiction
"A Higher Loyalty: Truth, Lies, and Leadership",James Comey,4.7,5983,3,2018,Non Fiction
"""

In [11]:
val booksPath = "data/books.csv"
val books = spark.read.option("header", "true").csv("data/books.csv")
books.show(5, false)

+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|Name                                                              |Author                  |User Rating|Reviews|Price|Year|Genre      |
+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|10-Day Green Smoothie Cleanse                                     |JJ Smith                |4.7        |17350  |8    |2016|Non Fiction|
|11/22/63: A Novel                                                 |Stephen King            |4.6        |2052   |22   |2011|Fiction    |
|12 Rules for Life: An Antidote to Chaos                           |Jordan B. Peterson      |4.7        |18979  |15   |2018|Non Fiction|
|1984 (Signet Classics)                                            |George Orwell           |4.7        |21424  |6    |2017|Fiction    |
|5,000 Awesome Facts (About Everything!) 

booksPath: String = "data/books.csv"
books: DataFrame = [Name: string, Author: string ... 5 more fields]

In [12]:
books.printSchema

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)



In [13]:
val booksSchema = StructType(
      StructField("Name", StringType, nullable = true) ::
        StructField("Author", StringType, nullable = true) ::
        StructField("User Rating", DoubleType, nullable = true) ::
        StructField("Reviews", IntegerType, nullable = true) ::
        StructField("Price", IntegerType, nullable = true) ::
        StructField("Year", IntegerType, nullable = true) ::
        StructField("Genre", StringType, nullable = true) :: Nil
    )

booksSchema: StructType = StructType(
  StructField("Name", StringType, true, {}),
  StructField("Author", StringType, true, {}),
  StructField("User Rating", DoubleType, true, {}),
  StructField("Reviews", IntegerType, true, {}),
  StructField("Price", IntegerType, true, {}),
  StructField("Year", IntegerType, true, {}),
  StructField("Genre", StringType, true, {})
)

In [14]:
val booksSchemaDF = spark.read.option("header", "true").schema(booksSchema).csv("data/books.csv")
booksSchemaDF.printSchema

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: double (nullable = true)
 |-- Reviews: integer (nullable = true)
 |-- Price: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Genre: string (nullable = true)



booksSchemaDF: DataFrame = [Name: string, Author: string ... 5 more fields]

In [15]:
booksSchemaDF.show(5, false)

+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|Name                                                              |Author                  |User Rating|Reviews|Price|Year|Genre      |
+------------------------------------------------------------------+------------------------+-----------+-------+-----+----+-----------+
|10-Day Green Smoothie Cleanse                                     |JJ Smith                |4.7        |17350  |8    |2016|Non Fiction|
|11/22/63: A Novel                                                 |Stephen King            |4.6        |2052   |22   |2011|Fiction    |
|12 Rules for Life: An Antidote to Chaos                           |Jordan B. Peterson      |4.7        |18979  |15   |2018|Non Fiction|
|1984 (Signet Classics)                                            |George Orwell           |4.7        |21424  |6    |2017|Fiction    |
|5,000 Awesome Facts (About Everything!) 

In [16]:
val booksInferDF = spark.read.option("header", "true").option("inferSchema ", "true").csv("data/books.csv")
booksInferDF.printSchema

root
 |-- Name: string (nullable = true)
 |-- Author: string (nullable = true)
 |-- User Rating: string (nullable = true)
 |-- Reviews: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Genre: string (nullable = true)



booksInferDF: DataFrame = [Name: string, Author: string ... 5 more fields]

## JSON

### Пример 1

In [17]:
"head -5 data/customer.json".!!

res17: String = """{"CustomerID":"12346","Country":"United Kingdom","Address":"Unit 1047 Box 4089\nDPO AA 57348","Birthdate":"1994-02-20 00:46:27","Email":"cooperalexis@hotmail.com","Name":"Lindsay Cowan","Username":"valenciajennifer"}
{"CustomerID":"12347","Country":"Iceland","Address":"55711 Janet Plaza Apt. 865\nChristinachester, CT 62716","Birthdate":"1988-06-21 00:15:34","Email":"timothy78@hotmail.com","Name":"Katherine David","Username":"hillrachel"}
{"CustomerID":"12348","Country":"Finland","Address":"Unit 2676 Box 9352\nDPO AA 38560","Birthdate":"1974-11-26 15:30:20","Email":"tcrawford@gmail.com","Name":"Leslie Martinez","Username":"serranobrian"}
{"CustomerID":"12349","Country":"Italy","Address":"2765 Powers Meadow\nHeatherfurt, CT 53165","Birthdate":"1977-05-06 23:57:35","Email":"dustin37@yahoo.com","Name":"Brad Cardenas","Username":"charleshudson"}
{"CustomerID":"12350","Country":"Norway","Address":"17677 Mark Crest\nWalterberg, IA 39017","Birthdate":"1996-09-13 19:14:27","E

In [18]:
("head -1 data/customer.json" #| "jq").!!

res18: String = """{
  "CustomerID": "12346",
  "Country": "United Kingdom",
  "Address": "Unit 1047 Box 4089\nDPO AA 57348",
  "Birthdate": "1994-02-20 00:46:27",
  "Email": "cooperalexis@hotmail.com",
  "Name": "Lindsay Cowan",
  "Username": "valenciajennifer"
}
"""

In [19]:
val customer = spark.read.json("data/customer.json")
customer.printSchema

root
 |-- Address: string (nullable = true)
 |-- Birthdate: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Email: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- Username: string (nullable = true)



customer: DataFrame = [Address: string, Birthdate: string ... 5 more fields]

In [20]:
customer.show(5, false)

+------------------------------------------------------+-------------------+--------------+----------+------------------------+---------------+----------------+
|Address                                               |Birthdate          |Country       |CustomerID|Email                   |Name           |Username        |
+------------------------------------------------------+-------------------+--------------+----------+------------------------+---------------+----------------+
|Unit 1047 Box 4089\nDPO AA 57348                      |1994-02-20 00:46:27|United Kingdom|12346     |cooperalexis@hotmail.com|Lindsay Cowan  |valenciajennifer|
|55711 Janet Plaza Apt. 865\nChristinachester, CT 62716|1988-06-21 00:15:34|Iceland       |12347     |timothy78@hotmail.com   |Katherine David|hillrachel      |
|Unit 2676 Box 9352\nDPO AA 38560                      |1974-11-26 15:30:20|Finland       |12348     |tcrawford@gmail.com     |Leslie Martinez|serranobrian    |
|2765 Powers Meadow\nHeatherfurt, 

### Пример 2

In [21]:
"head -20 data/countries.json".!!

res21: String = """[
    {
        "name": {
            "common": "Aruba",
            "official": "Aruba",
            "native": {
                "nld": {
                    "official": "Aruba",
                    "common": "Aruba"
                },
                "pap": {
                    "official": "Aruba",
                    "common": "Aruba"
                }
            }
        },
        "tld": [
            ".aw"
        ],
        "cca2": "AW",
"""

In [22]:
val countries = spark
                .read
                .format("json")
                .option("multiLine", "true")
                .load("data/countries.json")

countries: DataFrame = [altSpellings: array<string>, area: double ... 21 more fields]

In [23]:
countries.show(5, false)

+-----------------------------------------------+---------+------------------------------+------------+----+----+----+----+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [24]:
countries.printSchema

root
 |-- altSpellings: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- area: double (nullable = true)
 |-- borders: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- capital: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- cca2: string (nullable = true)
 |-- cca3: string (nullable = true)
 |-- ccn3: string (nullable = true)
 |-- cioc: string (nullable = true)
 |-- currencies: struct (nullable = true)
 |    |-- AED: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- AFN: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- ALL: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol: string (nullable = true)
 |    |-- AMD: struct (nullable = true)
 |    |    |-- name: string (nullable = true)
 |    |    |-- symbol:

## Parquet

In [25]:
"parquet-tools".!!

res25: String = """usage: parquet-tools [-h] {show,csv,inspect} ...

parquet CLI tools

positional arguments:
  {show,csv,inspect}
    show              Show human readable format. see `show -h`
    csv               Cat csv style. see `csv -h`
    inspect           Inspect parquet file. see `inspect -h`

options:
  -h, --help          show this help message and exit
"""

In [26]:
val sc = spark.sparkContext

sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@6554461f

In [27]:
val squaresDF = sc.makeRDD(1 to 5).map(i => (i, i * i)).toDF("value", "square")
squaresDF.show

+-----+------+
|value|square|
+-----+------+
|    1|     1|
|    2|     4|
|    3|     9|
|    4|    16|
|    5|    25|
+-----+------+



squaresDF: DataFrame = [value: int, square: int]

In [28]:
squaresDF.write.mode(SaveMode.Overwrite).parquet("data/test_table/key=1")

In [29]:
val cubesDF = spark.sparkContext.makeRDD(6 to 10).map(i => (i, i * i * i)).toDF("value", "cube")
cubesDF.show

+-----+----+
|value|cube|
+-----+----+
|    6| 216|
|    7| 343|
|    8| 512|
|    9| 729|
|   10|1000|
+-----+----+



cubesDF: DataFrame = [value: int, cube: int]

In [30]:
cubesDF.write.mode(SaveMode.Overwrite).parquet("data/test_table/key=2")

In [31]:
val mergedDF = spark.read.option("mergeSchema", "true").parquet("data/test_table")
mergedDF.printSchema

root
 |-- value: integer (nullable = true)
 |-- square: integer (nullable = true)
 |-- cube: integer (nullable = true)
 |-- key: integer (nullable = true)



mergedDF: DataFrame = [value: int, square: int ... 2 more fields]

In [32]:
mergedDF.show

+-----+------+----+---+
|value|square|cube|key|
+-----+------+----+---+
|    1|     1|NULL|  1|
|    2|     4|NULL|  1|
|    4|    16|NULL|  1|
|    5|    25|NULL|  1|
|    3|     9|NULL|  1|
|    6|  NULL| 216|  2|
|    9|  NULL| 729|  2|
|    8|  NULL| 512|  2|
|    7|  NULL| 343|  2|
|   10|  NULL|1000|  2|
+-----+------+----+---+



In [33]:
mergedDF.write.mode(SaveMode.Overwrite).parquet("data/merged.parquet")

In [34]:
val unMergedDF = spark.read.option("mergeSchema", "false").parquet("data/test_table")
unMergedDF.printSchema

root
 |-- value: integer (nullable = true)
 |-- square: integer (nullable = true)
 |-- key: integer (nullable = true)



unMergedDF: DataFrame = [value: int, square: int ... 1 more field]

In [35]:
unMergedDF.show

+-----+------+---+
|value|square|key|
+-----+------+---+
|    1|     1|  1|
|    2|     4|  1|
|    4|    16|  1|
|    5|    25|  1|
|    3|     9|  1|
|    6|  NULL|  2|
|    9|  NULL|  2|
|    8|  NULL|  2|
|    7|  NULL|  2|
|   10|  NULL|  2|
+-----+------+---+



In [36]:
"parquet-tools show data/merged.parquet".!!

res36: String = """+---------+----------+--------+-------+
|   value |   square |   cube |   key |
|---------+----------+--------+-------|
|       1 |        1 |    nan |     1 |
|       2 |        4 |    nan |     1 |
|       4 |       16 |    nan |     1 |
|       5 |       25 |    nan |     1 |
|       3 |        9 |    nan |     1 |
|       6 |      nan |    216 |     2 |
|       9 |      nan |    729 |     2 |
|       8 |      nan |    512 |     2 |
|       7 |      nan |    343 |     2 |
|      10 |      nan |   1000 |     2 |
+---------+----------+--------+-------+
"""

In [37]:
"ls -1 data/merged.parquet".!!

res37: String = """part-00000-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet
part-00001-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet
part-00002-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet
part-00003-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet
part-00004-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet
_SUCCESS
"""

In [38]:
val mergedFiles = "ls -1 data/merged.parquet".!!.split("\n").filter(_ != "_SUCCESS")

mergedFiles: Array[String] = Array(
  "part-00000-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet",
  "part-00001-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet",
  "part-00002-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet",
  "part-00003-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet",
  "part-00004-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet"
)

In [39]:
val file1 = mergedFiles(0)

s"parquet-tools inspect data/merged.parquet/$file1".!!

file1: String = "part-00000-c639dfeb-5ca8-48d3-9ea4-2b764b457adc-c000.snappy.parquet"
res39_1: String = """
############ file meta data ############
created_by: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
num_columns: 4
num_rows: 2
num_row_groups: 1
format_version: 1.0
serialized_size: 838


############ Columns ############
value
square
cube
key

############ Column(value) ############
name: value
path: value
max_definition_level: 1
max_repetition_level: 0
physical_type: INT32
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -5%)

############ Column(square) ############
name: square
path: square
max_definition_level: 1
max_repetition_level: 0
physical_type: INT32
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -6%)

############ Column(cube) ############
name: cube
...

In [40]:
val key2Files = "ls -1 data/test_table/key=2".!!.split("\n").filter(_ != "_SUCCESS")

key2Files: Array[String] = Array(
  "part-00000-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet",
  "part-00001-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet",
  "part-00003-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet",
  "part-00004-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet",
  "part-00006-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet",
  "part-00007-58d5b6f2-3ade-4885-a26d-6bef2060fa6f-c000.snappy.parquet"
)

In [41]:
s"parquet-tools inspect data/test_table/key=2/${key2Files(1)}".!!

res41: String = """
############ file meta data ############
created_by: parquet-mr version 1.13.1 (build db4183109d5b734ec5930d870cdae161e408ddba)
num_columns: 2
num_rows: 1
num_row_groups: 1
format_version: 1.0
serialized_size: 536


############ Columns ############
value
cube

############ Column(value) ############
name: value
path: value
max_definition_level: 0
max_repetition_level: 0
physical_type: INT32
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -7%)

############ Column(cube) ############
name: cube
path: cube
max_definition_level: 0
max_repetition_level: 0
physical_type: INT32
logical_type: None
converted_type (legacy): NONE
compression: SNAPPY (space_saved: -7%)

"""

## ORC

In [42]:
"java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar" !

cmd42.sc:1: postfix operator ! should be enabled
by making the implicit value scala.language.postfixOps visible.
----
This can be achieved by adding the import clause 'import scala.language.postfixOps'
or by setting the compiler option -language:postfixOps.
See the Scaladoc for value scala.language.postfixOps for a discussion
why the feature should be explicitly enabled.
val res42 = "java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar" !
                                                            ^
ORC Java Tools

usage: java -jar orc-tools-*.jar [--help] [--define X=Y] <command> <args>

Commands:
   check - check the index of the specified column
   convert - convert CSV/JSON/ORC files to ORC
   count - recursively find *.orc and print the number of rows
   data - print the data from the ORC file
   json-schema - scan JSON files to determine their schema
   key - print information about the keys
   merge - merge multiple ORC files into a single ORC file
   meta - print the metadata about th

res42: Int = 1

### Пример 1

In [43]:
"java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar data data/users.orc".!!

Processing data file data/users.orc [length: 547]


res43: String = """{"name":"Alyssa","favorite_color":null,"favorite_numbers":[3,9,15,20]}
{"name":"Ben","favorite_color":"red","favorite_numbers":[]}
________________________________________________________________________________________________________________________

"""

In [44]:
"java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar meta data/users.orc".!!

Processing data file data/users.orc [length: 547]


res44: String = """Structure for data/users.orc
File Version: 0.12 with ORC_135 by ORC Java 
Rows: 2
Compression: SNAPPY
Compression size: 262144
Calendar: Julian/Gregorian
Type: struct<name:string,favorite_color:string,favorite_numbers:array<int>>

Stripe Statistics:
  Stripe 1:
    Column 0: count: 2 hasNull: false
    Column 1: count: 2 hasNull: false bytesOnDisk: 18 min: Alyssa max: Ben sum: 9
    Column 2: count: 1 hasNull: true bytesOnDisk: 17 min: red max: red sum: 3
    Column 3: count: 2 hasNull: false bytesOnDisk: 6
    Column 4: count: 4 hasNull: false bytesOnDisk: 8 min: 3 max: 20 sum: 47

File Statistics:
  Column 0: count: 2 hasNull: false
  Column 1: count: 2 hasNull: false bytesOnDisk: 18 min: Alyssa max: Ben sum: 9
  Column 2: count: 1 hasNull: true bytesOnDisk: 17 min: red max: red sum: 3
  Column 3: count: 2 hasNull: false bytesOnDisk: 6
  Column 4: count: 4 hasNull: false bytesOnDisk: 8 min: 3 max: 20 sum: 47

Stripes:
  Stripe: offset: 3 data: 49 rows: 2 tail: 101 

In [45]:
val users = spark.read.orc("data/users.orc")
users.printSchema

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



users: DataFrame = [name: string, favorite_color: string ... 1 more field]

In [46]:
users.show

+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          NULL|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



### Пример 2

In [47]:
"java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar data data/example.orc".!!

Processing data file data/example.orc [length: 490]


res47: String = """{"one":-1.0,"two":"foo","three":true}
{"one":NaN,"two":"bar","three":false}
{"one":2.5,"two":"baz","three":true}
________________________________________________________________________________________________________________________

"""

In [48]:
"java17 -jar /opt/orc/orc-tools-2.3.0-uber.jar meta data/example.orc".!!

Processing data file data/example.orc [length: 490]


res48: String = """Structure for data/example.orc
File Version: 0.12 with ORC_CPP_ORIGINAL by ORC C++ 2.1.0
Rows: 3
Compression: NONE
Calendar: Julian/Gregorian
Type: struct<one:double,two:string,three:boolean>

Stripe Statistics:
  Stripe 1:
    Column 0: count: 3 hasNull: false
    Column 1: count: 3 hasNull: false min: -1.0 max: 2.5 sum: NaN
    Column 2: count: 3 hasNull: false min: bar max: foo sum: 9
    Column 3: count: 3 hasNull: false true: 2

File Statistics:
  Column 0: count: 3 hasNull: false
  Column 1: count: 3 hasNull: false min: -1.0 max: 2.5 sum: NaN
  Column 2: count: 3 hasNull: false min: bar max: foo sum: 9
  Column 3: count: 3 hasNull: false true: 2

Stripes:
  Stripe: offset: 3 data: 37 rows: 3 tail: 93 index: 93
    Stream: column 0 section ROW_INDEX start: 3 length 8
    Stream: column 1 section ROW_INDEX start: 11 length 40
    Stream: column 2 section ROW_INDEX start: 51 length 27
    Stream: column 3 section ROW_INDEX start: 78 length 18
    Stream: column 1 

In [49]:
val example = spark.read.orc("data/example.orc")
example.printSchema

root
 |-- one: double (nullable = true)
 |-- two: string (nullable = true)
 |-- three: boolean (nullable = true)



example: DataFrame = [one: double, two: string ... 1 more field]

In [50]:
example.show

+----+---+-----+
| one|two|three|
+----+---+-----+
|-1.0|foo| true|
| NaN|bar|false|
| 2.5|baz| true|
+----+---+-----+



## Avro

In [51]:
"java -jar /opt/avro/avro-tools-1.12.1.jar" !

cmd51.sc:1: postfix operator ! should be enabled
by making the implicit value scala.language.postfixOps visible.
----
This can be achieved by adding the import clause 'import scala.language.postfixOps'
or by setting the compiler option -language:postfixOps.
See the Scaladoc for value scala.language.postfixOps for a discussion
why the feature should be explicitly enabled.
val res51 = "java -jar /opt/avro/avro-tools-1.12.1.jar" !
                                                        ^
Version 1.12.1
 of Apache Avro
Copyright 2010-2015 The Apache Software Foundation

This product includes software developed at
The Apache Software Foundation (https://www.apache.org/).
----------------
Available tools:
    canonical  Converts an Avro Schema to its canonical form
          cat  Extracts samples from files
      compile  Generates Java code for the given schema.
       concat  Concatenates avro files without re-compressing.
        count  Counts the records in avro files or folders
  finger

res51: Int = 1

In [52]:
"java -jar /opt/avro/avro-tools-1.12.1.jar getschema data/users.avro".!!

26/06/28 15:59:31 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


res52: String = """{
  "type" : "record",
  "name" : "User",
  "namespace" : "example.avro",
  "fields" : [ {
    "name" : "name",
    "type" : "string"
  }, {
    "name" : "favorite_color",
    "type" : [ "string", "null" ]
  }, {
    "name" : "favorite_numbers",
    "type" : {
      "type" : "array",
      "items" : "int"
    }
  } ]
}
"""

In [53]:
"java -jar /opt/avro/avro-tools-1.12.1.jar tojson data/users.avro".!!

26/06/28 15:59:32 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


res53: String = """{"name":"Alyssa","favorite_color":null,"favorite_numbers":[3,9,15,20]}
{"name":"Ben","favorite_color":{"string":"red"},"favorite_numbers":[]}
"""

In [54]:
import $ivy.`org.apache.spark::spark-avro:3.5.8`

import $ivy.$

In [55]:
val usersDF = spark.read.format("avro").load("data/users.avro")
usersDF.printSchema

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)
 |-- favorite_numbers: array (nullable = true)
 |    |-- element: integer (containsNull = true)



usersDF: DataFrame = [name: string, favorite_color: string ... 1 more field]

In [56]:
usersDF.show

+------+--------------+----------------+
|  name|favorite_color|favorite_numbers|
+------+--------------+----------------+
|Alyssa|          NULL|  [3, 9, 15, 20]|
|   Ben|           red|              []|
+------+--------------+----------------+



In [57]:
usersDF
    .select("name", "favorite_color")
    .write
    .mode(SaveMode.Overwrite)
    .format("avro")
    .save("data/namesAndFavColors.avro")

In [58]:
val avroFiles = ("ls -1 data/namesAndFavColors.avro" #| "grep -v _SUCCESS").!!
val avroFile = avroFiles.split("\n")(0)

s"java -jar /opt/avro/avro-tools-1.12.1.jar getmeta data/namesAndFavColors.avro/$avroFile".!!

26/06/28 15:59:34 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


avroFiles: String = """part-00000-022223fe-4916-4815-b165-cbaaf7724b28-c000.avro
"""
avroFile: String = "part-00000-022223fe-4916-4815-b165-cbaaf7724b28-c000.avro"
res58_2: String = """avro.schema	{"type":"record","name":"topLevelRecord","fields":[{"name":"name","type":["string","null"]},{"name":"favorite_color","type":["string","null"]}]}
org.apache.spark.version	3.5.8
avro.codec	snappy
"""

In [59]:
val namesAndFavColorsDF = spark.read.format("avro").load("data/namesAndFavColors.avro")
namesAndFavColorsDF.printSchema

root
 |-- name: string (nullable = true)
 |-- favorite_color: string (nullable = true)



namesAndFavColorsDF: DataFrame = [name: string, favorite_color: string]

In [60]:
namesAndFavColorsDF.show

+------+--------------+
|  name|favorite_color|
+------+--------------+
|Alyssa|          NULL|
|   Ben|           red|
+------+--------------+



In [61]:
"java -jar /opt/avro/avro-tools-1.12.1.jar compile schema data/user.avsc data".!!

Input files to compile:
  data/user.avsc


res61: String = ""

In [62]:
"head data/example/avro/User.java".!!

res62: String = """/*
 * Autogenerated by Avro
 *
 * DO NOT EDIT DIRECTLY
 */
package example.avro;

import org.apache.avro.specific.SpecificData;
import org.apache.avro.util.Utf8;
import org.apache.avro.message.BinaryMessageEncoder;
"""